In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import torch
import torch.nn as nn
from hedging.envs import HedgeCallBS, HedgeCallHeston
from hedging.logit_normal import LogitNormal
from hedging.reward_utils import compute_discounted_cumsum_rewards
from hedging.plot_utils import plot_portfolio_vs_option_price


# Policy Gradient (Simple MLP) - BSM

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(
        self, 
        input_dim, 
        hidden_size, 
        action_dim=1, 
        log_std_min=-10, 
        log_std_max=2
    ):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        
        
        self.fc_log_std = nn.Linear(hidden_size, action_dim)
        
        self.action_dim = action_dim
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

    def forward(self, history_features):
        x = history_features[:, -1, :]
        x = torch.tanh(self.fc1(x))
        mu = torch.sigmoid(self.fc_mu(x))  # Directly bound mu
        log_std = self.fc_log_std(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mu, log_std

    def sample_action(self, mu, dist_params, deterministic=False):
        
        log_std = dist_params
        std = torch.exp(log_std)
        distribution = torch.distributions.Normal(mu, std)          
        if deterministic:
            action = mu
        else:
            action = distribution.rsample() # rsample let's you backprop
            action = torch.clamp(action, min=1e-6, max=1-1e-6)
        log_prob = distribution.log_prob(action).sum(-1) # action_dim > 1, you must sum to get the correct scalar log-prob for your policy loss.
        
        return action, log_prob



In [11]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])
num_paths = 100
num_steps = 250

env = HedgeCallBS(
    S0, K, maturity, r, sigma, num_paths, num_steps, history_len=1
)

# --- Policy Network Parameters ---
input_dim = 11
hidden_size = 64
history_len = 1

policy_net = PolicyNetwork(input_dim, hidden_size)

# --- Optimization Parameters ---
learning_rate = 1e-4 # Slower but more stable
# learning_rate = 1e-3 # Faster but less stable

optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)

# --- Other Parameters ---
num_episodes = 200
num_epochs = 10
discount_factor = 0.999

In [12]:
# Train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        log_prob_history = []
        reward_history = []
        state_history = []

        state, _ = env.reset(seed=epoch+1000) # avoid using seed=0 as it's for testing
        state_history.append(state)

        while True:
            policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
            policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(
                device
            )
            action_mu, action_sigma = policy_net(policy_net_input)
            action, log_prob = policy_net.sample_action(action_mu, action_sigma)
            log_prob_history.append(log_prob)
            action_np = action.detach().cpu().numpy()
            state, reward, done, _, _ = env.step(action_np)
            reward_history.append(reward)
            state_history.append(state)

            if all(done):
                break

        R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)
        R = R - R.mean(axis=1, keepdims=True)
        R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
        R = torch.tensor(R, dtype=torch.float32).to(device)
        optimizer.zero_grad()
        loss = (-R * torch.stack(log_prob_history)).mean()
        loss.backward()
        optimizer.step()

        if (episode + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss.item()}, Avg. Reward: {np.array(reward_history).mean()}"
            )

Epoch 1/10, Episode 10/200, Loss: 0.001851279754191637, Avg. Reward: -4.2002431810595775
Epoch 1/10, Episode 20/200, Loss: 0.001863529090769589, Avg. Reward: -4.383347388201601
Epoch 1/10, Episode 30/200, Loss: 0.0002787357661873102, Avg. Reward: -4.336974108979443
Epoch 1/10, Episode 40/200, Loss: 0.00028992656734772027, Avg. Reward: -4.315038529149681
Epoch 1/10, Episode 50/200, Loss: -0.004796941764652729, Avg. Reward: -4.311293965070828
Epoch 1/10, Episode 60/200, Loss: -0.006977887358516455, Avg. Reward: -4.306258081981836
Epoch 1/10, Episode 70/200, Loss: -0.008429543115198612, Avg. Reward: -4.238636884129571
Epoch 1/10, Episode 80/200, Loss: -0.008535408414900303, Avg. Reward: -4.220522389722442
Epoch 1/10, Episode 90/200, Loss: -0.011114751920104027, Avg. Reward: -4.076782007270121
Epoch 1/10, Episode 100/200, Loss: -0.008812449872493744, Avg. Reward: -4.158235683843616
Epoch 1/10, Episode 110/200, Loss: -0.011146614328026772, Avg. Reward: -4.040757107369645
Epoch 1/10, Episode

In [13]:
action_sigma.min(), action_sigma.max(), action_sigma.mean(), action.min(), action.max(), action.mean()


(tensor(0.0029, grad_fn=<MinBackward1>),
 tensor(4.4828, grad_fn=<MaxBackward1>),
 tensor(0.6491, grad_fn=<MeanBackward0>),
 tensor(5.1712e-06, grad_fn=<MinBackward1>),
 tensor(1.0000, grad_fn=<MaxBackward1>),
 tensor(0.8312, grad_fn=<MeanBackward0>))

In [14]:
rewards = np.array(reward_history)
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(-75.21535972448166,
 -3.2498217166221366e-06,
 -5.402938206398209,
 7.536675890641707)

In [15]:
# Test

env = HedgeCallBS(
    S0, K, maturity, r, sigma, 5, num_steps
)

log_prob_history = []
reward_history = []
state_history = []

state, _ = env.reset(seed=0)
#state = state[:, None, :]
state_history.append(state)

while True:
    policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
    policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
    action_mu, action_sigma = policy_net(policy_net_input)
    action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
    log_prob_history.append(log_prob)
    action_np = action.detach().cpu().numpy()
    state, reward, done, _, _ = env.step(action_np)
    reward_history.append(reward)
    #state = state[:, None, :]
    state_history.append(state)

    if all(done):
        break

In [16]:
rewards = np.array(reward_history)
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(-43.870934849591684,
 -0.0005569581720052952,
 -4.142376839954368,
 5.127194291308451)

In [17]:
action.min()

tensor(0.0008, grad_fn=<MinBackward1>)

In [18]:
plot_portfolio_vs_option_price(env)

# Policy Gradient (Simple MLP) - Heston

In [24]:
class PolicyNetwork(nn.Module):
    def __init__(
        self, 
        input_dim, 
        hidden_size, 
        action_dim=1, 
        log_std_min=-10, 
        log_std_max=2
    ):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        
        
        self.fc_log_std = nn.Linear(hidden_size, action_dim)
        
        self.action_dim = action_dim
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

    def forward(self, history_features):
        x = history_features[:, -1, :]
        x = torch.tanh(self.fc1(x))
        mu = torch.sigmoid(self.fc_mu(x))  # Directly bound mu
        log_std = self.fc_log_std(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mu, log_std

    def sample_action(self, mu, dist_params, deterministic=False):
        
        log_std = dist_params
        std = torch.exp(log_std)
        distribution = torch.distributions.Normal(mu, std)          
        if deterministic:
            action = mu
        else:
            action = distribution.rsample() # rsample let's you backprop
            action = torch.clamp(action, min=1e-6, max=1-1e-6)
        log_prob = distribution.log_prob(action).sum(-1) # action_dim > 1, you must sum to get the correct scalar log-prob for your policy loss.
        
        return action, log_prob



In [25]:
class PolicyNetwork(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_size,
        action_dim=1,
    ):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        self.fc_sigma = nn.Linear(hidden_size, action_dim)
        self.softplus = nn.Softplus()

    def forward(self, history_features):

        x = history_features[
            :, -1, :
        ]  # simple MLP just uses the latest state's feature
        x = torch.tanh(self.fc1(x))
        mu = self.fc_mu(x)
        sigma = self.softplus(self.fc_sigma(x))
        return mu, sigma

    def sample_action(self, mu, sigma, deterministic=False):
        logit_normal = LogitNormal(mu, sigma)

        if deterministic:
            action = torch.sigmoid(mu)
            log_prob = None
        else:
            action = logit_normal.rsample()
            log_prob = logit_normal.log_prob(action)

        return action, log_prob

In [26]:
# --- Env Parameters ---

S0 = np.array([100, 120, 80])
K = np.array([
    [90, 100, 110],
    [100, 120, 140],
    [70, 80, 90]
])
v0 = np.array([0.05, 0.04, 0.06])
r = 0.03
tau = 0.5
trap = 1

params = {
    "kappa": np.array([5.0, 2.5, 3.0]),
    "theta": np.array([0.05, 0.035, 0.045]),
    "rho": np.array([-0.8, -0.6, -0.5]),
    "sigma": np.array([0.5, 0.4, 0.55]),
    "lda": np.array([0.0, 0.0, 0.0]) # not ued right now
}

env = HedgeCallHeston(
    S0=S0, K = K, r=r, v0=v0, theta=params["theta"], rho=params["rho"],
    kappa=params["kappa"], xi=params["sigma"], maturity=tau,
    num_steps=250, num_paths=100, history_len=1
)


In [27]:
# --- Policy Network Parameters ---
input_dim = 11
hidden_size = 64
history_len = 1

policy_net = PolicyNetwork(input_dim, hidden_size)

# --- Optimization Parameters ---
learning_rate = 5e-5 # Slower but more stable
# learning_rate = 1e-3 # Faster but less stable

optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)

# --- Other Parameters ---
num_episodes = 200
num_epochs = 10
discount_factor = 0.999

In [28]:
# Train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        log_prob_history = []
        reward_history = []
        state_history = []

        state, _ = env.reset(seed=epoch+1000) # avoid using seed=0 as it's for testing
        state_history.append(state)

        while True:
            policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
            policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(
                device
            )
            action_mu, action_sigma = policy_net(policy_net_input)
            action, log_prob = policy_net.sample_action(action_mu, action_sigma)
            log_prob_history.append(log_prob)
            action_np = action.detach().cpu().numpy() # Is this good enough?
            state, reward, done, _, _ = env.step(action_np)
            reward_history.append(reward)
            state_history.append(state)

            if all(done):
                break

        R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)
        R = R - R.mean(axis=1, keepdims=True)
        R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
        R = torch.tensor(R, dtype=torch.float32).to(device)
        optimizer.zero_grad()
        loss = (-R * torch.stack(log_prob_history)).mean()
        loss.backward()
        optimizer.step()

        if (episode + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss.item()}, Avg. Reward: {np.array(reward_history).mean()}"
            )

: 

In [ ]:
# Test

env = HedgeCallHeston(
    S0=S0, K = K, r=r, v0=v0, theta=params["theta"], rho=params["rho"],
    kappa=params["kappa"], xi=params["sigma"], maturity=tau,
    num_steps=100, num_paths=1, history_len=1
)

log_prob_history = []
reward_history = []
state_history = []

state, _ = env.reset(seed=0)
#state = state[:, None, :]
state_history.append(state)

while True:
    policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
    policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
    action_mu, action_sigma = policy_net(policy_net_input)
    action, log_prob = policy_net.sample_action(action_mu, action_sigma, False)
    log_prob_history.append(log_prob)
    action_np = action.detach().cpu().numpy()
    state, reward, done, _, _ = env.step(action_np)
    reward_history.append(reward)
    #state = state[:, None, :]
    state_history.append(state)

    if all(done):
        break

In [ ]:
rewards = np.array(reward_history)
rewards.mean(), rewards.std(), rewards.max(), rewards.min()

(-3.00910752761254,
 2.957850023344163,
 -0.0003397393511459512,
 -13.798516305945633)